# Solution C2 · Tree instability: 30 resamples, 30 different trees

**Type**: Break the model

## Objective

Measure the **variance** of trees: resampling with the bootstrap, trees with the same parameters disagree on a large fraction of the examples. See how depth and minimum leaf size reduce the disagreement, and that the **majority vote** of 30 unlimited trees is much more accurate
than the single tree (the idea of forests).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(1500, n_features=10, n_informative=4, n_redundant=0, class_sep=0.8, flip_y=0.1, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
configs = {"no limits": {}, "max_depth=5": {"max_depth": 5}, "max_depth=3": {"max_depth": 3},
           "min_samples_leaf=20": {"min_samples_leaf": 20}, "min_samples_leaf=50": {"min_samples_leaf": 50}}

## Commented solution

In [ ]:
rng = np.random.default_rng(0)
predictions, leaves = {}, {}
for name, params in configs.items():
    P, F = [], []
    for _ in range(30):
        i = rng.integers(0, len(X_tr), len(X_tr))                                # bootstrap sample of the training set
        tree = DecisionTreeClassifier(random_state=0, **params).fit(X_tr[i], y_tr[i])
        P.append(tree.predict(X_te)); F.append(tree.get_n_leaves())
    predictions[name], leaves[name] = np.array(P), np.array(F)
print({k: v.shape for k, v in predictions.items()})

In [ ]:
table = pd.DataFrame(index=list(configs), columns=["mean_acc", "acc_std", "disagreement", "mean_leaves"], dtype=float)
for name in configs:
    P = predictions[name]
    accuracies = (P == y_te).mean(1)
    table.loc[name] = [accuracies.mean(), accuracies.std(),
                       np.mean([(P[i] != P[j]).mean() for i in range(15) for j in range(i + 1, 15)]), leaves[name].mean()]
table.round(3)

In [ ]:
vote = (predictions["no limits"].mean(0) > 0.5).astype(int)
vote_acc = float((vote == y_te).mean())
vote_gain = vote_acc - table.loc["no limits", "mean_acc"]
disagreement_reduction = table.loc["no limits", "disagreement"] / table.loc["max_depth=3", "disagreement"]
print(f"accuracy of the single tree without limits: {table.loc['no limits', 'mean_acc']:.3f} | vote of 30 trees: {vote_acc:.3f} (+{vote_gain:.3f})")
print(f"disagreement between trees without limits / with max_depth=3: {disagreement_reduction:.1f} times")

## Check

In [ ]:
with verify():
    check(set(predictions) == set(configs) and all(np.asarray(p).shape == (30, len(y_te)) for p in predictions.values()), "30 predictions on the test set for each configuration", "one row per tree, one column per test example")
    check(table.notna().all().all() and table.loc["no limits", "mean_acc"] < 0.76 and table.loc["max_depth=3", "mean_acc"] > 0.75, "the single trees without limits are below 0.76; with max_depth=3 above 0.75", "mean accuracy of the 30 trees")
    check(table.loc["no limits", "disagreement"] > 0.2 and table.loc["max_depth=3", "disagreement"] < 0.13 and table.loc["min_samples_leaf=50", "disagreement"] < 0.13, "without limits two trees disagree on over 20% of the examples; with max_depth=3 or leaf 50 on less than 13%", "mean over the pairs of trees")
    check(table.loc["no limits", "mean_leaves"] > 60 and table.loc["max_depth=3", "mean_leaves"] <= 8, "without limits over 60 leaves; max_depth=3 at most 8", "get_n_leaves()")
    check(vote_acc is not None and vote_acc > 0.78 and vote_gain > 0.04, "the vote of 30 trees without limits goes over 0.78: over 4 points above the single tree", "mean of the predictions > 0.5")
    check(disagreement_reduction > 1.8, "the disagreement without limits is over 1.8 times that of max_depth=3", "compare the two rows")
done()

## What you learned

- **Two trees without limits trained on bootstrap samples of the same training set disagree on about a quarter of the examples** (24%): with the same parameters, on the same problem. The root may stay almost always the same (it is a dominant feature), but the deeper branches follow the noise of the sample.
- **Regularizing reduces the disagreement by more than two times** (11% with `max_depth=3`, 8% with `min_samples_leaf=50`) and raises the mean accuracy (0.74 → 0.77): the same thing from the point of view of the variance. The price is a less flexible model (more bias), and here it is a price worth paying.
- **The majority vote of 30 trees without limits beats the accuracy of every single tree** and also of the regularized trees (0.80 against 0.77): averaging removes the "random" errors of each tree and the systematic ones remain. It is exactly the recipe of **bagging** and of random forests (module 8): deep trees (low bias, high variance) with the variance knocked down by the average.
- Note the cost: 30 trees with over 80 leaves each lose the readability of the single tree. Interpretability and accuracy are in tension here.

## Common pitfall

Interpreting the details of a single deep tree (a threshold, the order of the branches) as the structure of the phenomenon: changing a hundred examples changes a quarter of the predictions. If you need stability, regularize or average many trees.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why does a tree have high variance and how do you reduce it?</summary>

Because every split depends on few examples and the choices cascade: a small change in the data moves a threshold and changes the whole subtree. You reduce it by regularizing (minimum leaf size, depth, pruning) or by averaging many trees trained on different samples (bagging,
random forest): the average of estimators that are little biased but very variable has a much smaller variance, without increasing the bias.

</details>